# CNN-LSTM — GAN augmentation sweep (Interpretation B, 5-seed Monte Carlo)

In [1]:
import pandas as pd, numpy as np, torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix)
import sys; sys.path.insert(0, "../")
from util import *
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", torch.cuda.get_device_name(device) if "cuda" in device else device)

from models.cnn_lstm import CNN_LSTM_v1, CNN_LSTM_v2
MODEL_CLS = CNN_LSTM_v2
MODEL_NAME = MODEL_CLS.__name__

SCENES = range(1, 7)          # scenes 1..6 (scene 0 = full data, no augmentation)
RATIOS = [0.0, 0.5, 1.0, 2.0] # 0.0 = no-augmentation baseline
SEEDS = [1, 2, 3, 4, 5]
EPOCHS, LR, WD, USE_SCHED = 70, 1e-2, 1e-4, True


device: NVIDIA L4


In [2]:
def to_tensors(arr):
    X = torch.from_numpy(arr[:, 1:14].astype("float32")).unsqueeze(1)
    y = torch.from_numpy(arr[:,  -1].astype("int64"))
    return X, y


def load_scenario(scenario_dir):
    train = pd.read_csv(scenario_dir + "train.csv").to_numpy()
    val   = pd.read_csv(scenario_dir + "val.csv").to_numpy()
    test  = pd.read_csv(scenario_dir + "test.csv").to_numpy()
    X_tr_n, X_va_n, X_te_n, _, _ = preprocess_scenario(train, val, test)
    return to_tensors(X_tr_n), to_tensors(X_va_n), to_tensors(X_te_n)


def make_model(model_cls, device, seed=0, **kwargs):
    """Fresh CNN-LSTM with Xavier init for Conv/Linear; default PyTorch init
    for LSTM (MATLAB's Glorot applies to Conv and FC, LSTM uses its own)."""
    torch.manual_seed(seed)
    model = model_cls(n_classes=8, **kwargs).to(device)
    for m in model.modules():
        if isinstance(m, nn.Conv1d):
            nn.init.xavier_uniform_(m.weight)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)
            nn.init.zeros_(m.bias)
    return model


def make_loaders(X_tr, y_tr, X_va, y_va, X_te, y_te, batch_size=50, seed=0):
    g = torch.Generator().manual_seed(seed)
    perm = torch.randperm(len(X_tr), generator=g)
    X_tr_s, y_tr_s = X_tr[perm], y_tr[perm]
    train_loader = DataLoader(TensorDataset(X_tr_s, y_tr_s),
                              batch_size=batch_size, shuffle=False)
    val_loader   = DataLoader(TensorDataset(X_va, y_va), batch_size=batch_size)
    test_loader  = DataLoader(TensorDataset(X_te, y_te), batch_size=batch_size)
    return train_loader, val_loader, test_loader


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = total_correct = total_n = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss    += loss.item() * xb.size(0)
        total_correct += (logits.argmax(1) == yb).sum().item()
        total_n       += xb.size(0)
    return total_loss / total_n, total_correct / total_n


@torch.no_grad()
def evaluate_loader(model, loader, criterion, device):
    model.eval()
    total_loss = total_correct = total_n = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        total_loss    += criterion(logits, yb).item() * xb.size(0)
        total_correct += (logits.argmax(1) == yb).sum().item()
        total_n       += xb.size(0)
    return total_loss / total_n, total_correct / total_n


@torch.no_grad()
def predict_all(model, loader, device):
    model.eval()
    y_true, y_pred = [], []
    for xb, yb in loader:
        logits = model(xb.to(device))
        y_pred.append(logits.argmax(1).cpu().numpy())
        y_true.append(yb.numpy())
    return np.concatenate(y_true), np.concatenate(y_pred)

def run_scenario(scenario_idx, scenario_dir, model_cls, device,
                 epochs=70, lr=1e-2, weight_decay=1e-4, seed=0, verbose=True):
    (X_tr, y_tr), (X_va, y_va), (X_te, y_te) = load_scenario(scenario_dir)
    train_loader, val_loader, test_loader = make_loaders(
        X_tr, y_tr, X_va, y_va, X_te, y_te, batch_size=50, seed=seed)

    model = make_model(model_cls, device, seed=seed)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr,
                           betas=(0.9, 0.999), eps=1e-8,
                           weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

    # ---- compute metrics setup ----
    n_params, params_by_type = count_parameters(model)
    reset_gpu_peak_memory(device)

    if verbose:
        print(f"\n=== Scenario {scenario_idx} ({model_cls.__name__}) ===")
        print(f"  params: {n_params:,}  breakdown: {params_by_type}")

    # ---- TRAINING with timer ----
    with Timer(device) as train_timer:
        for ep in range(1, epochs + 1):
            tr_loss, tr_acc = train_one_epoch(model, train_loader,
                                              criterion, optimizer, device)
            va_loss, va_acc = evaluate_loader(model, val_loader,
                                              criterion, device)
            scheduler.step()
            if verbose and (ep == 1 or ep % 10 == 0 or ep == epochs):
                print(f"  ep {ep:3d} | train loss {tr_loss:.4f} acc {tr_acc:.3f}"
                      f" | val loss {va_loss:.4f} acc {va_acc:.3f}")

    peak_mem_mb = get_gpu_peak_memory_mb(device)

    # ---- INFERENCE timing ----
    X_te_dev = X_te.to(device)
    def _predict(X):
        model.eval()
        with torch.no_grad():
            return model(X).argmax(1)
    inf_stats = measure_inference_time(_predict, X_te_dev, device,
                                       n_warmup=5, n_runs=20)

    # ---- TEST accuracy ----
    y_true, y_pred = predict_all(model, test_loader, device)
    acc = accuracy_score(y_true, y_pred)
    p, r, f, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(8)))

    if verbose:
        print(f"  TEST: acc={acc:.4f}  P={p:.4f}  R={r:.4f}  F1={f:.4f}")
        print(f"  COMPUTE: train={train_timer.elapsed:.1f}s  "
              f"inf={inf_stats['per_sample_ms']:.3f}ms/sample  "
              f"peak_mem={peak_mem_mb:.1f}MB")

    return {
        "scenario":    scenario_idx,
        "model":       model_cls.__name__,
        "n_train":     len(X_tr),
        "accuracy":    acc,
        "precision":   p,
        "recall":      r,
        "f1":          f,
        "confusion":   cm,
        # ---- new compute fields ----
        "n_params":    n_params,
        "train_sec":   round(train_timer.elapsed, 2),
        "inf_ms_per_sample": round(inf_stats["per_sample_ms"], 4),
        "peak_mem_mb": round(peak_mem_mb, 1),
    }


# ---------------- Interpretation B: array-based runner ----------------
def tensors_from_arrays(train_arr, val_arr, test_arr):
    X_tr_n, X_va_n, X_te_n, _, _ = preprocess_scenario(train_arr, val_arr, test_arr)
    return to_tensors(X_tr_n), to_tensors(X_va_n), to_tensors(X_te_n)


def run_from_arrays(train_arr, val_arr, test_arr, model_cls, device,
                    epochs=EPOCHS, lr=LR, weight_decay=WD,
                    use_scheduler=USE_SCHED, seed=0):
    (X_tr, y_tr), (X_va, y_va), (X_te, y_te) = tensors_from_arrays(
        train_arr, val_arr, test_arr)
    train_loader, val_loader, test_loader = make_loaders(
        X_tr, y_tr, X_va, y_va, X_te, y_te, batch_size=50, seed=seed)

    model = make_model(model_cls, device, seed=seed)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = (optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)
                 if use_scheduler else None)

    for ep in range(1, epochs + 1):
        train_one_epoch(model, train_loader, criterion, optimizer, device)
        evaluate_loader(model, val_loader, criterion, device)
        if scheduler is not None:
            scheduler.step()

    y_true, y_pred = predict_all(model, test_loader, device)
    p, r, f, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0)
    return {"accuracy": accuracy_score(y_true, y_pred),
            "precision": p, "recall": r, "f1": f, "n_train": len(X_tr)}


In [3]:
from GANs.dcgans import train_class_gans, augment

val_arr  = pd.read_csv("../CSV_Files/eval/val.csv").to_numpy()
test_arr = pd.read_csv("../CSV_Files/eval/test.csv").to_numpy()

rows, gan_rows = [], []
for seed in SEEDS:
    for scene in SCENES:
        train_arr = pd.read_csv(f"../CSV_Files/seed{seed}/scene{scene}/train.csv").to_numpy()

        # Train the 8 per-class DCGANs ONCE per (seed, scene); reuse across ratios
        gans, gan_hist = train_class_gans(train_arr, seed=seed, device=device)
        for c, h in gan_hist.items():
            gan_rows.append({"seed": seed, "scene": scene, "class": c,
                             "d_acc_final":  round(h["d_acc"][-1], 4),
                             "d_loss_final": round(h["d_loss"][-1], 4),
                             "g_loss_final": round(h["g_loss"][-1], 4)})
        mean_dacc = float(np.mean([h["d_acc"][-1] for h in gan_hist.values()]))
        print(f">> seed {seed} scene {scene}: mean final D_acc = {mean_dacc:.3f}  "
            f"(near 0.5 = faithful, near 1.0 = generator collapsed)")


        for ratio in RATIOS:
            aug = augment(train_arr, gans, ratio, seed=seed, device=device)
            res = run_from_arrays(aug, val_arr, test_arr, MODEL_CLS, device, seed=seed)
            res.update(seed=seed, scene=scene, ratio=ratio)
            rows.append(res)
            print(f"seed {seed} scene {scene} ratio {ratio}: "
                  f"F1={res['f1']:.4f} (n_train={res['n_train']})")

df = pd.DataFrame(rows)
df.to_csv(f"{MODEL_NAME.lower()}_gan_results_raw.csv", index=False)
pd.DataFrame(gan_rows).to_csv(f"{MODEL_NAME.lower()}_gan_diagnostics.csv", index=False)

agg = df.groupby(["scene", "ratio"])["f1"].agg(["mean", "std"]).reset_index()
print(agg.to_string(index=False))
agg.to_csv(f"{MODEL_NAME.lower()}_gan_results_agg.csv", index=False)


>> seed 1 scene 1: mean final D_acc = 0.607  (near 0.5 = faithful, near 1.0 = generator collapsed)
seed 1 scene 1 ratio 0.0: F1=0.8824 (n_train=7866)
seed 1 scene 1 ratio 0.5: F1=0.9486 (n_train=11790)
seed 1 scene 1 ratio 1.0: F1=0.9048 (n_train=15785)
seed 1 scene 1 ratio 2.0: F1=0.9313 (n_train=23839)
>> seed 1 scene 2: mean final D_acc = 0.582  (near 0.5 = faithful, near 1.0 = generator collapsed)
seed 1 scene 2 ratio 0.0: F1=0.8373 (n_train=3922)
seed 1 scene 2 ratio 0.5: F1=0.8850 (n_train=5815)
seed 1 scene 2 ratio 1.0: F1=0.8214 (n_train=7694)
seed 1 scene 2 ratio 2.0: F1=0.7881 (n_train=11628)
>> seed 1 scene 3: mean final D_acc = 0.534  (near 0.5 = faithful, near 1.0 = generator collapsed)
seed 1 scene 3 ratio 0.0: F1=0.8372 (n_train=789)
seed 1 scene 3 ratio 0.5: F1=0.7970 (n_train=1169)
seed 1 scene 3 ratio 1.0: F1=0.7399 (n_train=1556)
seed 1 scene 3 ratio 2.0: F1=0.7559 (n_train=2317)
>> seed 1 scene 4: mean final D_acc = 0.571  (near 0.5 = faithful, near 1.0 = generator 